**Cell #01**

# RAG11 Nutrition — Stage 5: Hypothetical Document Embeddings (HyDE) Examples

Adds **HyDE (Hypothetical Document Embeddings)** as a third retrieval mode
alongside plain vector search and hybrid search: instead of embedding the
bare question, Claude first drafts a short hypothetical textbook-style
paragraph that would answer it, and *that paragraph* is embedded and
searched with instead. Demonstrated with 3 nutrition-specific examples
chosen to show where a bare-question embedding struggles and HyDE helps.

## The problem, in one picture

A question and a textbook answer are written in different "shapes" of
English. Questions are short and interrogative; the real chunks in this
database are long, declarative, technical prose. Say a nutrition RAG gets
asked *"Does drinking coffee before exercise dehydrate you enough to hurt
performance?"*:

- Pure vector search embeds that question directly. It's great at "roughly
  the same topic" — but the question and the real answer chunk ("Caffeine
  has a mild diuretic effect, but studies show that moderate pre-exercise
  intake does not cause clinically significant dehydration...") don't
  always sit close together in embedding space, even though the chunk is
  exactly right, simply because they're written in very different registers.
- **HyDE's fix:** first ask Claude to draft a short hypothetical paragraph
  that *would* answer the question — it doesn't need to be correct, it only
  needs to plausibly use the same vocabulary real chunks do ("diuretic
  effect," "habituated users") — then embed *that paragraph* instead of the
  question, and search with it.
- Why it helps: the hypothetical paragraph is written in the same
  style/vocabulary as the real textbook chunks, so it's more likely to land
  near the actual correct textbook paragraph in vector space than the bare
  question would.

Think of it like: instead of asking a librarian "where's the fiber book?",
you hand them a page that looks like the page you want, and say "find me
more like this."

## Prerequisites

**No SQL migration needed** — unlike hybrid search, HyDE needs **zero**
schema change: it calls the exact same `match_rag11_child_chunks` RPC plain
`retrieve_chunks()` already uses. The only difference is *which text* gets
embedded before that call.

This notebook gets its retrieval/generation logic from `./reusable_code/`
— the same package every other `stage2_ask_examples*.ipynb` imports from —
so all five notebooks share one implementation of `ask_question`
(`reusable_code/generation.py`), `retrieve_chunks`
(`reusable_code/retrieval.py`), and `retrieve_chunks_hyde`
(`reusable_code/hypothetical_document_embedding.py`).
See `documentation/HOW_IT_WORKS_Hypothetical_Document_Embedding.html` for
the full write-up and `reusable_code/README.md` for the one-paragraph
summary.

In [1]:
from reusable_code import (
    init_clients,
    ask_question,
    retrieve_chunks,
    embed_query,
    generate_hypothetical_document,
    embed_hypothetical_document,
    retrieve_chunks_hyde,
    rerank_chunks,
    expand_to_parent_chunks,
    NUM_CONTEXT_CHUNKS,
    HYDE_MAX_TOKENS,
    EMBEDDING_MODEL,
    GENERATION_MODEL,
)

clients = init_clients()
print("Clients ready.")
print("Embedding model:", EMBEDDING_MODEL, "| Generation model:", GENERATION_MODEL,
      "| HyDE max tokens:", HYDE_MAX_TOKENS)

Clients ready.
Embedding model: voyage-3 | Generation model: claude-sonnet-5 | HyDE max tokens: 200


**Cell #03**

## A helper to see bare-question vs. HyDE retrieval side by side

`show_hyde_comparison()` runs both against the real database: a plain
dense search (bare-question embedding) and `retrieve_chunks_hyde()`
(Claude drafts a hypothetical answer, which is embedded and searched with
instead) — printed together, along with the hypothetical paragraph itself,
so the effect of HyDE is visible rather than assumed.

In [2]:
def show_hyde_comparison(question: str, top_n: int = NUM_CONTEXT_CHUNKS):
    """Run plain (bare-question) dense search and HyDE (hypothetical-document)
    dense search for `question` and print both side by side, along with the
    hypothetical paragraph Claude drafted. Returns (plain, hyde,
    hypothetical_doc) so the caller can inspect or reuse any of them."""
    print(f"Q: {question}\n")

    hyde, hypothetical_doc = retrieve_chunks_hyde(
        question, match_count=top_n, return_hypothetical_document=True
    )
    print("-- Hypothetical document Claude drafted (a retrieval probe only -- never shown to a user) --")
    print(f"  {hypothetical_doc}\n")

    plain = retrieve_chunks(question, match_count=top_n)
    print(f"-- Plain retrieval top {len(plain)} (bare-question embedding, cosine distance, closest first) --")
    for i, row in enumerate(plain, start=1):
        preview = row["rowJSON"]["text"].replace("\n", " ")[:100]
        source = row["rowJSON"].get("source_key", "?")
        print(f"  {i:>2}. dist={row['cosine_distance']:.4f}  [{source}]  {preview}...")

    print(f"\n-- HyDE retrieval top {len(hyde)} (hypothetical-document embedding, cosine distance, closest first) --")
    for i, row in enumerate(hyde, start=1):
        preview = row["rowJSON"]["text"].replace("\n", " ")[:100]
        source = row["rowJSON"].get("source_key", "?")
        print(f"  {i:>2}. dist={row['cosine_distance']:.4f}  [{source}]  {preview}...")

    plain_top_ids = {r["rowGUID"] for r in plain}
    hyde_found_something_plain_missed = any(r["rowGUID"] not in plain_top_ids for r in hyde)
    print(f"\nHyDE surfaced a top-{top_n} chunk plain (bare-question) retrieval alone would have missed: "
          f"{hyde_found_something_plain_missed}")
    print("-" * 80)
    return plain, hyde, hypothetical_doc


example_results = {}   # question -> (plain, hyde, hypothetical_doc), filled in by the 3 examples below

**Cell #05**

## Three nutrition examples for the HyDE demonstration

Each chosen for a different reason a bare-question embedding struggles on
its own:

1. **A conversationally-phrased question with no exact textbook
   terminology to key off of** — "does coffee mess with your workout?" in
   plainer words, worded as a casual yes/no worry rather than a clinical
   question.
2. **An indirect, causal "why" question whose mechanism is stated in very
   different words in the source text** — the real chunk likely discusses
   "compensatory hyperphagia" and hormonal appetite signaling, vocabulary
   the question itself never uses.
3. **A broad, practical "how much should I do" question whose real answer
   lives in precise clinical figures and terminology** — "how much protein
   to build muscle" vs. a chunk phrased around "resistance-trained
   individuals" and "muscle protein synthesis."


In [3]:
EXAMPLE_QUESTIONS = [
    # 1. Conversationally phrased, no exact textbook vocabulary to key off of.
    "Does drinking coffee before exercise dehydrate you enough to hurt performance?",
    # 2. Indirect/causal phrasing -- the mechanism is in the text, but not in these words.
    "Why does skipping breakfast make some people overeat later in the day?",
    # 3. Broad practical question whose answer lives in precise clinical terms.
    "How much protein should someone eat if they want to build muscle?",
]

**Cell #07**

### Example 1 — a conversationally-phrased question, no exact terminology

In [4]:
example_results[EXAMPLE_QUESTIONS[0]] = show_hyde_comparison(EXAMPLE_QUESTIONS[0])

Q: Does drinking coffee before exercise dehydrate you enough to hurt performance?

-- Hypothetical document Claude drafted (a retrieval probe only -- never shown to a user) --
  Caffeine's mild diuretic effect, mediated through adenosine receptor antagonism in the renal tubules, transiently increases urine output, but habitual coffee consumers develop tolerance to this natriuretic and diuretic response within a few days of regular intake. Controlled trials comparing caffeinated coffee ingestion (3–6 mg/kg body mass) against water or placebo prior to exercise consistently show no significant differences in total body water, plasma volume, urine output over 24 hours, or thermoregulatory strain during subsequent activity. The fluid volume delivered by a standard cup of coffee

-- Plain retrieval top 5 (bare-question embedding, cosine distance, closest first) --
   1. dist=0.2978  [source11]  [Source: _OceanofPDF.com_Nancy_Clarks_Sports_Nutrition_Guidebook_-_Nancy_Clark.pdf | Section: Flui

**Cell #09**

### Example 2 — an indirect "why" question, mechanism stated in different words

In [5]:
example_results[EXAMPLE_QUESTIONS[1]] = show_hyde_comparison(EXAMPLE_QUESTIONS[1])

Q: Why does skipping breakfast make some people overeat later in the day?

-- Hypothetical document Claude drafted (a retrieval probe only -- never shown to a user) --
  Skipping breakfast disrupts the diurnal regulation of appetite-related hormones, producing compensatory hyperphagia later in the day in susceptible individuals. Following an overnight fast, ghrelin levels normally rise and are then suppressed by morning food intake; when breakfast is omitted, ghrelin secretion remains elevated and prolonged, sustaining orexigenic signaling to the hypothalamic arcuate nucleus well into the late morning and afternoon. Concurrently, the absence of an early meal delays the postprandial rise in peptide YY (PYY) and glucagon-like pe

-- Plain retrieval top 5 (bare-question embedding, cosine distance, closest first) --
   1. dist=0.4094  [source11]  [Source: _OceanofPDF.com_Nancy_Clarks_Sports_Nutrition_Guidebook_-_Nancy_Clark.pdf | Section: Breakf...
   2. dist=0.4095  [source11]  [Source: _

**Cell #11**

### Example 3 — a broad practical question, precise clinical answer

In [6]:
example_results[EXAMPLE_QUESTIONS[2]] = show_hyde_comparison(EXAMPLE_QUESTIONS[2])

Q: How much protein should someone eat if they want to build muscle?

-- Hypothetical document Claude drafted (a retrieval probe only -- never shown to a user) --
  Muscle protein synthesis (MPS) is maximally stimulated by protein intakes in the range of 1.6 to 2.2 g per kilogram of body weight per day for individuals engaged in regular resistance training, a threshold substantially higher than the Recommended Dietary Allowance of 0.8 g/kg established for sedentary populations to prevent deficiency. Distributing this intake across four to five meals, each providing approximately 0.3 to 0.4 g/kg or roughly 20 to 40 g of high-quality protein, appears to optimize the anabolic response by repeatedly surpassing the le

-- Plain retrieval top 5 (bare-question embedding, cosine distance, closest first) --
   1. dist=0.3003  [source11]  [Source: _OceanofPDF.com_Nancy_Clarks_Sports_Nutrition_Guidebook_-_Nancy_Clark.pdf | Section: Protei...
   2. dist=0.3093  [source11]  [Source: _OceanofPDF.com

**Cell #13**

## `ask_question(..., use_hyde=...)` end to end

`use_hyde` is an optional keyword argument on `ask_question` — it defaults
to `False`, so every existing call in `stage2_ask_examples1/2/3/4.ipynb`
behaves exactly as before. Passing `use_hyde=True` swaps in
`retrieve_chunks_hyde()` as the source of candidate chunks instead of plain
`retrieve_chunks()`. It also **composes** with `use_rerank=True` and
`expand_to_parents=True` from the earlier Stage 2 notebooks — HyDE only
changes what gets embedded before the dense search call, so everything
downstream of retrieval works exactly the same. (It is *not* combined with
`use_hybrid=True` — hybrid search's dense half always embeds the raw
question.)

In [ ]:
demo_question = EXAMPLE_QUESTIONS[0]

baseline = ask_question(demo_question, match_count=NUM_CONTEXT_CHUNKS, use_hyde=False)
hyde_answer = ask_question(demo_question, match_count=NUM_CONTEXT_CHUNKS, use_hyde=True)
hyde_rerank_expand_answer = ask_question(
    demo_question, match_count=NUM_CONTEXT_CHUNKS,
    use_hyde=True, use_rerank=True, expand_to_parents=True,
)

print("Q:", demo_question)

print("\n-- ask_question(..., use_hyde=False) [default, bare-question embedding] --")
print("chunks used:", baseline["chunks_used"], "| candidates considered:", baseline["candidates_considered"])
if baseline["short_answer"]:
    print("Short answer:", baseline["short_answer"])
print(baseline["answer"])

print("\n-- ask_question(..., use_hyde=True) --")
print("chunks used:", hyde_answer["chunks_used"], "| candidates considered:", hyde_answer["candidates_considered"])
print("hypothetical document used:", hyde_answer["hypothetical_document"])
if hyde_answer["short_answer"]:
    print("Short answer:", hyde_answer["short_answer"])
print(hyde_answer["answer"])

print("\n-- ask_question(..., use_hyde=True, use_rerank=True, expand_to_parents=True) [full pipeline] --")
print("chunks used:", hyde_rerank_expand_answer["chunks_used"],
      "| candidates considered:", hyde_rerank_expand_answer["candidates_considered"])
print("source pages:", hyde_rerank_expand_answer["source_pages"])
if hyde_rerank_expand_answer["short_answer"]:
    print("Short answer:", hyde_rerank_expand_answer["short_answer"])
print(hyde_rerank_expand_answer["answer"])

**Cell #15**

## The HyDE building blocks in isolation — draft, then embed

`generate_hypothetical_document()` and `embed_hypothetical_document()` are
small, independently callable functions — `retrieve_chunks_hyde()` just
composes them with the same dense RPC `retrieve_chunks()` calls. Both need
a real network call (Claude to draft, Voyage to embed), so unlike
`reciprocal_rank_fusion()` or `expand_to_parent_chunks()` in the earlier
notebooks, there's no toy-data version — but it's still worth calling them
directly to see exactly what gets embedded, and how that differs from
`embed_query()`'s bare-question embedding.

In [ ]:
isolation_question = "What role does vitamin D play in calcium absorption?"

hypothetical_doc = generate_hypothetical_document(isolation_question)
print("Hypothetical document Claude drafted:\n")
print(f"  {hypothetical_doc}\n")

hyde_embedding = embed_hypothetical_document(hypothetical_doc)
print(f"HyDE embedding (input_type='document'): {len(hyde_embedding)} dims, first 5 = {hyde_embedding[:5]}")

query_embedding = embed_query(isolation_question)
print(f"For comparison, embed_query() on the bare question (input_type='query'): "
      f"{len(query_embedding)} dims, first 5 = {query_embedding[:5]}")

**Cell #17**

## Summary across the 3 examples

Same idea as the hybrid-search notebook's summary table: a quick scan of
how many chunks each method returned, and whether HyDE surfaced something
plain (bare-question) retrieval alone would have missed within the
top-N.

In [ ]:
print(f"{'#':<3} {'plain hits':<11} {'hyde hits':<10} {'hyde found plain missed?':<26} question")
for i, question in enumerate(EXAMPLE_QUESTIONS, start=1):
    plain, hyde, hypothetical_doc = example_results[question]
    plain_ids = {r["rowGUID"] for r in plain}
    hyde_found_something_new = bool(hyde) and any(r["rowGUID"] not in plain_ids for r in hyde)
    print(f"{i:<3} {len(plain):<11} {len(hyde):<10} {str(hyde_found_something_new):<26} {question}")

**Cell #19**

## For stakeholders — what this means in plain terms

| Without HyDE | With HyDE |
| --- | --- |
| The bare question is embedded and searched with directly. | Claude first drafts a short hypothetical answer paragraph; *that* is embedded and searched with instead. |
| A casually- or indirectly-phrased question can rank the right chunk lower, since it "sounds different" from the technical answer text. | A question phrased in plain language retrieves about as well as one already phrased in textbook language — users don't have to "talk like the textbook." |
| Zero extra cost per question beyond the existing embedding call. | One extra, short (~150-200 token) Claude call per question, before retrieval. |
| No new moving parts. | No new moving parts either — same dense RPC, same table, same index; only *which text* gets embedded changes. |

**Net effect:** better retrieval on conversationally-phrased, indirect, or
broad practical questions — the kind that don't share much literal
vocabulary with the technical textbook prose that actually answers them —
at the cost of one extra short Claude call per question. A good trade
anywhere users won't naturally phrase questions the way a textbook would,
which is most real usage.

## For AI engineers — what this means technically

- **Zero schema change.** `retrieve_chunks_hyde()` calls the exact same
  `match_rag11_child_chunks` RPC plain `retrieve_chunks()` already uses —
  the only difference is *which text* gets embedded before that call
  (`reusable_code/hypothetical_document_embedding.py`).
- **`input_type='document'`, not `'query'`.** A HyDE paragraph is
  document-shaped text, so it's embedded on the same asymmetric side the
  real child chunks were embedded on in Stage 1.2 — see Cell #16 above for
  the side-by-side embedding call.
- **Composable, not a replacement.** `retrieve_chunks_hyde()` returns the
  exact same row shape `retrieve_chunks()` does, so `rerank_chunks()` and
  `expand_to_parent_chunks()` work on its output unchanged — see
  `ask_question(use_hyde=..., use_rerank=..., expand_to_parents=...)` in
  Cell #14 above.
- **Not combined with hybrid search here.** `ask_question(use_hyde=True,
  use_hybrid=True)` ignores `use_hyde` — hybrid search's dense half always
  embeds the raw question; combining a HyDE-derived dense ranking with
  hybrid's keyword ranking isn't wired in.
- **The hypothetical paragraph is a retrieval probe only.** It is never
  shown to the user and is not fact-checked — the real, grounded answer
  still comes from Claude reading the retrieved excerpts, exactly as
  before. `ask_question(...)["hypothetical_document"]` exposes it for
  debugging/inspection, and is `None` when `use_hyde=False`.

**Cell #21**

## Save workspace to GitHub

Synchronize this notebook and any code changes to GitHub (with auto lock
recovery and conflict resolution), the same helper
`stage2_ask_examples4_parent_chunk_expansion.ipynb` uses -- shared via
`reusable_code.save_to_github`.

In [ ]:
from reusable_code import save_to_github

save_to_github("stage2_ask_examples5_hypothetical_document_embedding.ipynb - HyDE examples added")